# 03｜準備 Retrain 資料集

輸入資料夾只需依真實類別分成 `retrain_raw/real` 與 `retrain_raw/fake`，
檔名不限。程式先切分 train、val、test，再用 YOLO 擷取最大人臉並輸出
`dataset_vit_retrain`。

原始圖片只讀取、不移動、不刪除。YOLO 找不到人臉的圖片會被跳過並記錄。


In [ ]:
# =========================
# 參數設定區
# =========================
RAW_RETRAIN_DIR = "retrain_raw"
OUTPUT_DIR = "dataset_vit_retrain"
YOLO_MODEL_PATH = "yolov11n-face.pt"

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
FACE_CONF = 0.25
FACE_CROP_SCALE = 1.15
RANDOM_SEED = 42

# True 會清除舊 OUTPUT_DIR；為避免誤刪，預設 False
OVERWRITE_OUTPUT = True
FAILED_CSV = "retrain_failed_images.csv"
SPLIT_MANIFEST_CSV = "retrain_split_manifest.csv"


In [ ]:
import math
import random
import shutil
from pathlib import Path

import cv2
import pandas as pd
from tqdm.auto import tqdm
from ultralytics import YOLO

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def validate_ratios(train_ratio, val_ratio, test_ratio):
    if not math.isclose(train_ratio + val_ratio + test_ratio, 1.0, abs_tol=1e-8):
        raise ValueError("TRAIN_RATIO、VAL_RATIO、TEST_RATIO 的總和必須為 1")


def prepare_output(path, overwrite):
    path = Path(path)
    if path.exists():
        if not overwrite:
            raise FileExistsError(
                f"{path} 已存在。確認可以清除後，將 OVERWRITE_OUTPUT 改成 True。"
            )
        shutil.rmtree(path)
    for split in ("train", "val", "test"):
        for label in ("fake", "real"):
            (path / split / label).mkdir(parents=True, exist_ok=True)


def list_class_images(root, label):
    folder = Path(root) / label
    if not folder.exists():
        raise FileNotFoundError(f"找不到類別資料夾：{folder}")
    return sorted(
        path for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def stratified_split(root, train_ratio, val_ratio, seed):
    rng = random.Random(seed)
    assignments = []
    for label in ("fake", "real"):
        paths = list_class_images(root, label)
        if not paths:
            raise ValueError(f"{label} 資料夾沒有圖片")
        rng.shuffle(paths)
        n = len(paths)
        train_end = int(n * train_ratio)
        val_end = train_end + int(n * val_ratio)
        for split, items in (
            ("train", paths[:train_end]),
            ("val", paths[train_end:val_end]),
            ("test", paths[val_end:]),
        ):
            assignments.extend(
                {"source_path": str(path), "label": label, "split": split}
                for path in items
            )
    return assignments


def largest_face_box(detector, image, confidence):
    results = detector.predict(image, conf=confidence, verbose=False)
    if not results or results[0].boxes is None or len(results[0].boxes) == 0:
        return None
    boxes = results[0].boxes.xyxy.detach().cpu().numpy()
    return max(boxes, key=lambda b: float((b[2] - b[0]) * (b[3] - b[1])))


def crop_square(image, box, scale):
    height, width = image.shape[:2]
    x1, y1, x2, y2 = map(float, box)
    center_x, center_y = (x1 + x2) / 2, (y1 + y2) / 2
    side = int(min(max(x2 - x1, y2 - y1) * scale, width, height))
    left = min(max(int(round(center_x - side / 2)), 0), width - side)
    top = min(max(int(round(center_y - side / 2)), 0), height - side)
    return image[top:top + side, left:left + side]


def unique_output_path(folder, source):
    candidate = folder / source.name
    counter = 1
    while candidate.exists():
        candidate = folder / f"{source.stem}_{counter}{source.suffix.lower()}"
        counter += 1
    return candidate


In [ ]:
# =========================
# 執行切分與裁臉
# =========================
validate_ratios(TRAIN_RATIO, VAL_RATIO, TEST_RATIO)
prepare_output(OUTPUT_DIR, OVERWRITE_OUTPUT)
assignments = stratified_split(
    RAW_RETRAIN_DIR, TRAIN_RATIO, VAL_RATIO, RANDOM_SEED
)
pd.DataFrame(assignments).to_csv(
    SPLIT_MANIFEST_CSV, index=False, encoding="utf-8-sig"
)

detector = YOLO(YOLO_MODEL_PATH)
failures = []
saved_counts = {}
for item in tqdm(assignments, desc="YOLO 裁臉"):
    source = Path(item["source_path"])
    split, label = item["split"], item["label"]
    image = cv2.imread(str(source))
    reason = None
    if image is None:
        reason = "read_failed"
    else:
        box = largest_face_box(detector, image, FACE_CONF)
        if box is None:
            reason = "no_face"
        else:
            face = crop_square(image, box, FACE_CROP_SCALE)
            if face.size == 0:
                reason = "crop_failed"

    if reason is not None:
        failures.append({**item, "reason": reason})
        continue

    destination = unique_output_path(
        Path(OUTPUT_DIR) / split / label, source
    )
    if not cv2.imwrite(str(destination), face):
        failures.append({**item, "reason": "write_failed"})
        continue
    key = f"{split}/{label}"
    saved_counts[key] = saved_counts.get(key, 0) + 1

pd.DataFrame(
    failures,
    columns=["source_path", "label", "split", "reason"],
).to_csv(FAILED_CSV, index=False, encoding="utf-8-sig")

print("完成：", saved_counts)
print(f"失敗圖片：{len(failures)} 張，紀錄於 {FAILED_CSV}")
